# LightForge: free GPU qualification on Colab or Kaggle

**Prepared experiment; no GPU result or 75% speedup claim is included.** Run the original public demo through all 27 Deux graphs and all 335 inference calls, preserving Float32 weights and output. The question is whether CUDA executes the expensive operations and produces output that qualifies for timing under the existing experiment rules.

This is the first separation-stage experiment. GAME, the complete vocal stage, network transfer, Android integration and whole-song elapsed time remain outside its scope. A 75% reduction means completing the same complete analysis in one quarter of the baseline time; this notebook cannot establish that target alone.

## Choose a free runtime

- **Colab:** open this notebook, choose **Runtime → Change runtime type → GPU** (a free T4 if offered), then connect. Do not select a paid upgrade. GPU supply and session duration vary.
- **Kaggle:** import this `.ipynb` into a private notebook, enable Internet, and select an available free GPU in notebook settings. Account verification or quota availability may be required. Use one notebook session; do not run duplicate experiments.
- Keep this an interactive notebook. There is no app server, SSH tunnel, distributed worker, automatic reconnect or quota workaround.

Run cells in order. Initial downloads are approximately **3.6 GB**; allow at least **12 GiB free disk** for extracted models, runtime libraries and evidence. No Drive mount, API token, private audio or signing key is needed. Download the evidence before the free session ends.

Sources: [Colab limits](https://research.google.com/colaboratory/faq.html), [Kaggle notebooks](https://www.kaggle.com/docs/notebooks), [ORT CUDA requirements](https://onnxruntime.ai/docs/execution-providers/CUDA-ExecutionProvider.html), [cuDNN 9.10.2 support matrix](https://docs.nvidia.com/deeplearning/cudnn/backend/v9.10.2/reference/support-matrix.html).

## 1. Configure an isolated workspace

The experiment source is frozen to commit `1c7a7d2d3bc0a9eab94bbcb8266710ea7f8ded0b`. Only required source files are checked out. Every experiment gets a fresh output directory; existing evidence is preserved. Python 3.11+ with safe tar extraction support and Linux x86-64 are required.

In [ ]:
import datetime, hashlib, importlib.util, json, os, platform, shutil, subprocess, sys
import signal, tarfile, tempfile, urllib.request, uuid, zipfile
from pathlib import Path

SOURCE_COMMIT = "1c7a7d2d3bc0a9eab94bbcb8266710ea7f8ded0b"
REPOSITORY = "https://github.com/CyberBASSLord-666/LightForge.git"
BASE = Path("/kaggle/working" if Path("/kaggle/working").is_dir() else "/content" if Path("/content").is_dir() else Path.cwd())
WORK = BASE / "lightforge-gpu-qualification"
REPO, TOOLCHAIN, ASSETS = (WORK / name for name in ("source", "toolchain", "assets"))
RUN = WORK / "evidence" / (datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8])
assert sys.version_info >= (3, 11), "Use a Python 3.11+ runtime."
assert hasattr(tarfile, "data_filter"), "This Python needs security updates supporting tarfile's data filter."
assert platform.system() == "Linux" and platform.machine() == "x86_64", "Linux x86-64 is required."
WORK.mkdir(parents=True, exist_ok=True)
RUN.mkdir(parents=True, exist_ok=False)
assert shutil.disk_usage(WORK).free >= 12 * 1024**3, "Need at least 12 GiB free disk."
print("Evidence directory:", RUN)
print("Python:", sys.version.split()[0], "Platform:", platform.platform())

## 2. Check the assigned GPU before downloading

This check records the actual GPU and driver. No GPU means stop here and select an available free GPU in the notebook UI. The later provider traces must still prove real CUDA arithmetic on every graph; detecting a GPU alone is not an inference result.

In [ ]:
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total,compute_cap", "--format=csv,noheader"],
                     capture_output=True, text=True, timeout=30)
(RUN / "gpu-inventory.txt").write_text(gpu.stdout + gpu.stderr)
assert gpu.returncode == 0 and gpu.stdout.strip(), "No usable NVIDIA GPU; select a free GPU runtime."
for line in gpu.stdout.strip().splitlines():
    name, driver, memory, capability = [value.strip() for value in line.split(",")]
    assert tuple(map(int, driver.split("."))) >= (525, 60, 13), "The pinned CUDA 12 stack needs a compatible driver."
    assert float(capability) >= 6.0, "The pinned cuDNN runtime requires compute capability 6.0 or later."
print(gpu.stdout.strip())
print("Inventory only; GPU execution and numerical equivalence remain untested.")

## 3. Retrieve the exact experiment source

No branch tip is used. The pinned source includes the benchmark's integrity checks, unchanged production Java source and original graph manifest.

In [ ]:
def command(arguments, **kwargs):
    return subprocess.run([str(value) for value in arguments], check=True, text=True, **kwargs)

patterns = ["/android/src/", "/android/native-runtime.json", "/tests/NativeDeuxExecutionBenchmark.java",
            "/tools/benchmark_deux_execution.py", "/tools/profile_deux_operators.py",
            "/tools/benchmark_deux_accelerator.py", "/tools/bootstrap_toolchain.py",
            "/web/analysis/models/deux/manifest.json"]
if not (REPO / ".git").exists():
    REPO.mkdir(parents=True, exist_ok=True)
    command(["git", "init", "-q", REPO])
    command(["git", "-C", REPO, "remote", "add", "origin", REPOSITORY])
assert command(["git", "-C", REPO, "remote", "get-url", "origin"], capture_output=True).stdout.strip() == REPOSITORY
head_check = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True)
if head_check.returncode != 0:
    # A first fetch may have been interrupted; retry into this empty checkout.
    command(["git", "-C", REPO, "config", "core.sparseCheckout", "true"])
    command(["git", "-C", REPO, "config", "remote.origin.promisor", "true"])
    command(["git", "-C", REPO, "config", "remote.origin.partialclonefilter", "blob:none"])
    (REPO / ".git/info/sparse-checkout").write_text("\n".join(patterns) + "\n")
    command(["git", "-C", REPO, "fetch", "--filter=blob:none", "--depth=1", "origin", SOURCE_COMMIT])
    command(["git", "-C", REPO, "checkout", "--detach", SOURCE_COMMIT])
head = command(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True).stdout.strip()
assert head == SOURCE_COMMIT, "Existing checkout differs; choose a new WORK path and rerun setup."
assert not command(["git", "-C", REPO, "status", "--porcelain", "--untracked-files=no"], capture_output=True).stdout.strip(), "Tracked source changed; choose a new WORK path."
print("Verified source commit:", head)

## 4. Define verified downloads

Downloads are streamed, limited to the declared size, verified with SHA-256 and atomically installed. Interrupted downloads remain unusable until downloaded and checked again. Cached complete files are always rehashed.

In [ ]:
def digest(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

def fetch(url, destination, size, sha256):
    destination = Path(destination)
    if destination.is_file() and not destination.is_symlink() and destination.stat().st_size == size and digest(destination) == sha256:
        print("Verified cached", destination.name)
        return destination
    assert not destination.is_symlink(), "Refusing a symlink destination."
    destination.parent.mkdir(parents=True, exist_ok=True)
    partial = destination.with_name(destination.name + ".partial")
    assert not partial.is_symlink(), "Refusing a symlink partial download."
    request = urllib.request.Request(url, headers={"User-Agent": "LightForge-public-GPU-experiment/1"})
    with urllib.request.urlopen(request, timeout=180) as response, partial.open("wb") as output:
        count = 0
        while block := response.read(1024 * 1024):
            count += len(block)
            if count > size:
                raise ValueError("Download exceeds pinned size: " + destination.name)
            output.write(block)
    assert partial.stat().st_size == size and digest(partial) == sha256, "Download digest mismatch: " + destination.name
    partial.replace(destination)
    print("Verified download", destination.name)
    return destination

def extract_member(archive, member, destination, size=None, sha256=None):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    assert not destination.is_symlink(), "Refusing a symlink destination."
    partial = destination.with_name(destination.name + ".partial")
    assert not partial.is_symlink(), "Refusing a symlink partial output."
    with archive.open(member) as source, partial.open("wb") as output:
        shutil.copyfileobj(source, output, 1024 * 1024)
    if size is not None:
        assert partial.stat().st_size == size, "Extracted size mismatch: " + member
    if sha256 is not None:
        assert digest(partial) == sha256, "Extracted digest mismatch: " + member
    partial.replace(destination)
    return destination

## 5. Recover the original graphs and public audio

The public 2.3.1 APK is used only as an asset container. Its whole-file digest is fixed; all 27 extracted graph files must also match the pinned source manifest. The demo is the shipped `glass-castle.wav`. No APK is rebuilt, signed, installed or published.

In [ ]:
APK_SIZE = 1205116958
APK_SHA = "af83bf403875c55d42fd695d43f6e193114899c1324fbaeaffd6c02d299d882f"
AUDIO_SHA = "33f07d502ba19832b62b97d8fb4a11354e81310264bd885d7a06438228773650"
MANIFEST_SHA = "6aebf45e6e7f6fa974f14fe47a252fc01f48da4815f40a1fdf10641a432529a9"
APK_URL = "https://github.com/CyberBASSLord-666/LightForge/releases/download/v2.3.1/LightForge-2.3.1.apk"
apk = fetch(APK_URL, ASSETS / "LightForge-2.3.1.apk", APK_SIZE, APK_SHA)
manifest_path = REPO / "web/analysis/models/deux/manifest.json"
assert digest(manifest_path) == MANIFEST_SHA
manifest = json.loads(manifest_path.read_text())
MODELS, AUDIO = ASSETS / "deux", ASSETS / "glass-castle.wav"
MODELS.mkdir(exist_ok=True)
with zipfile.ZipFile(apk) as archive:
    assert hashlib.sha256(archive.read("assets/analysis/models/deux/manifest.json")).hexdigest() == MANIFEST_SHA
    for name, pin in manifest["files"].items():
        assert Path(name).name == name, "Unexpected model path."
        extract_member(archive, "assets/analysis/models/deux/" + name, MODELS / name, pin["bytes"], pin["sha256"])
    extract_member(archive, "assets/demo/glass-castle.wav", AUDIO, sha256=AUDIO_SHA)
shutil.copyfile(manifest_path, MODELS / "manifest.json")
print("Verified", len(manifest["files"]), "original graphs and public demo.")

## 6. Install the pinned Java and ONNX runtimes locally

JDK17 and Android SDK35 compile the unmodified production Java snapshots. Only the Android platform JAR is needed; no Android build tools or application package are built. Dependencies stay within this notebook workspace.

In [ ]:
spec = importlib.util.spec_from_file_location("lightforge_bootstrap", REPO / "tools/bootstrap_toolchain.py")
bootstrap = importlib.util.module_from_spec(spec)
spec.loader.exec_module(bootstrap)
TOOLCHAIN.mkdir(exist_ok=True)
for pin in bootstrap.PACKAGES:
    if pin["name"] not in ("jdk17.tar.gz", "platform-35_r02.zip"):
        continue
    archive_path = fetch(pin["url"], TOOLCHAIN / "downloads" / pin["name"], pin["size"], pin["sha256"])
    if pin["name"] == "platform-35_r02.zip":
        with zipfile.ZipFile(archive_path) as archive:
            extract_member(archive, pin["source"] + "/android.jar", TOOLCHAIN / pin["target"] / "android.jar")
    else:
        # Re-extract verified bytes on each setup; do not trust a stale installed marker.
        with tempfile.TemporaryDirectory(dir=TOOLCHAIN) as temp:
            with tarfile.open(archive_path) as archive:
                archive.extractall(temp, filter="data")
            target = TOOLCHAIN / pin["target"]
            if target.exists():
                shutil.rmtree(target)
            shutil.move(str(Path(temp) / pin["source"]), target)

runtime = json.loads((REPO / "android/native-runtime.json").read_text())
host = runtime["host"]
fetch(host["url"], TOOLCHAIN / "onnx" / host["name"], host["bytes"], host["sha256"])
fetch("https://repo.maven.apache.org/maven2/com/microsoft/onnxruntime/onnxruntime_gpu/1.25.1/onnxruntime_gpu-1.25.1.jar",
      TOOLCHAIN / "onnx/onnxruntime_gpu-1.25.1.jar", 397558371,
      "0a22d140ee2a064944b7ee45b7f7a8deb113f58e9be514da85c6dfbe85262649")
fetch("https://repo.maven.apache.org/maven2/org/json/json/20260719/json-20260719.jar", TOOLCHAIN / "test-json.jar", 90031,
      "c243f45f9590c12694a4142ed3f07fc70dfb71e4daebd05ae234bf92a2da92a6")
command([TOOLCHAIN / "jdk17/bin/java", "-version"])
print("Pinned Java compilation and ORT dependencies ready.")

## 7. Install isolated CUDA 12 / cuDNN 9 libraries

These are exact Linux x86-64 wheels published by NVIDIA on PyPI, selected on September 20, 2026. No notebook Python packages or system drivers are replaced. The wheels are treated as verified archives of native libraries. CUDA12.9, cuDNN9.10.2 and a compatible driver are required; compatibility is confirmed by actual graph execution later.

The runtime pin remains fixed even if a newer package is published. cuDNN9.10.2 retains Pascal support; a P100 is allowed by the runtime preflight, but its performance and ORT graph execution are still unproven.

In [ ]:
CUDA_WHEELS = [('nvidia-cudnn-cu12',
  '9.10.2.21',
  'nvidia_cudnn_cu12-9.10.2.21-py3-none-manylinux_2_27_x86_64.whl',
  706758467,
  '949452be657fa16687d0930933f032835951ef0892b37d2d53824d1a84dc97a8',
  'ba/51/e123d997aa098c61d029f76663dedbfb9bc8dcf8c60cbd6adbe42f76d049'),
 ('nvidia-cublas-cu12',
  '12.9.1.4',
  'nvidia_cublas_cu12-12.9.1.4-py3-none-manylinux_2_27_x86_64.whl',
  581242350,
  '453611eb21a7c1f2c2156ed9f3a45b691deda0440ec550860290dc901af5b4c2',
  '77/3c/aa88abe01f3be3d1f8f787d1d33dc83e76fec05945f9a28fbb41cfb99cd5'),
 ('nvidia-cuda-runtime-cu12',
  '12.9.79',
  'nvidia_cuda_runtime_cu12-12.9.79-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl',
  3493179,
  '25bba2dfb01d48a9b59ca474a1ac43c6ebf7011f1b0b8cc44f54eb6ac48a96c3',
  'bc/46/a92db19b8309581092a3add7e6fceb4c301a3fd233969856a8cbf042cd3c'),
 ('nvidia-cuda-nvrtc-cu12',
  '12.9.86',
  'nvidia_cuda_nvrtc_cu12-12.9.86-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl',
  89568129,
  '210cf05005a447e29214e9ce50851e83fc5f4358df8b453155d5e1918094dcb4',
  'b8/85/e4af82cc9202023862090bfca4ea827d533329e925c758f0cde964cb54b7'),
 ('nvidia-nvjitlink-cu12',
  '12.9.86',
  'nvidia_nvjitlink_cu12-12.9.86-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl',
  39748338,
  'e3f1171dbdc83c5932a45f0f4c99180a70de9bd2718c1ab77d14104f6d7147f9',
  '46/0c/c75bbfb967457a0b7670b8ad267bfc4fffdf341c074e0a80db06c24ccfd4'),
 ('nvidia-curand-cu12',
  '10.3.10.19',
  'nvidia_curand_cu12-10.3.10.19-py3-none-manylinux_2_27_x86_64.whl',
  68295626,
  '49b274db4780d421bd2ccd362e1415c13887c53c214f0d4b761752b8f9f6aa1e',
  '31/44/193a0e171750ca9f8320626e8a1f2381e4077a65e69e2fb9708bd479e34a'),
 ('nvidia-cufft-cu12',
  '11.4.1.4',
  'nvidia_cufft_cu12-11.4.1.4-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl',
  200877592,
  'c67884f2a7d276b4b80eb56a79322a95df592ae5e765cf1243693365ccab4e28',
  '95/f4/61e6996dd20481ee834f57a8e9dca28b1869366a135e0d42e2aa8493bdd4')]
CUDA_ROOT = TOOLCHAIN / "cuda-wheels"
assert not CUDA_ROOT.is_symlink(), "Refusing a symlink CUDA directory."
if CUDA_ROOT.exists():
    shutil.rmtree(CUDA_ROOT)  # Recreate only this experiment's isolated native runtime.
for package, version, filename, size, sha256, location in CUDA_WHEELS:
    url = "https://files.pythonhosted.org/packages/" + location + "/" + filename
    wheel = fetch(url, TOOLCHAIN / "downloads" / filename, size, sha256)
    with zipfile.ZipFile(wheel) as archive:
        members = [name for name in archive.namelist() if name.startswith("nvidia/") and "/lib/" in name and ".so" in Path(name).name]
        assert members, "No native libraries found in " + filename
        for member in members:
            relative = Path(member)
            assert not relative.is_absolute() and ".." not in relative.parts
            extract_member(archive, member, CUDA_ROOT / relative)
library_dirs = sorted(str(path) for path in CUDA_ROOT.glob("nvidia/*/lib") if path.is_dir())
assert len(library_dirs) == len(CUDA_WHEELS)
benchmark_env = os.environ.copy()
benchmark_env["LD_LIBRARY_PATH"] = os.pathsep.join(library_dirs + ([os.environ["LD_LIBRARY_PATH"]] if os.environ.get("LD_LIBRARY_PATH") else []))
benchmark_env["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
benchmark_env["NVIDIA_TF32_OVERRIDE"] = "0"
assert not any(benchmark_env.get(key) for key in ("JAVA_TOOL_OPTIONS", "_JAVA_OPTIONS", "JDK_JAVA_OPTIONS")), "A Java option injection variable is set; use a clean runtime."
setup_receipt = {
    "schema": "lightforge.notebook-setup.v1", "sourceCommit": SOURCE_COMMIT,
    "createdUtc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "python": sys.version, "platform": platform.platform(), "publicApkSha256": APK_SHA,
    "publicAudioSha256": digest(AUDIO), "modelManifestSha256": digest(MODELS / "manifest.json"),
    "cudaWheelPins": [dict(package=p, version=v, filename=f, bytes=s, sha256=h,
                           source="https://pypi.org/project/" + p + "/" + v + "/") for p,v,f,s,h,_ in CUDA_WHEELS],
    "extractedNativeLibraries": {str(path.relative_to(CUDA_ROOT)): {"bytes": path.stat().st_size, "sha256": digest(path)}
                                for path in CUDA_ROOT.rglob("*") if path.is_file()},
    "qualityApproved": False, "target75Proven": False, "releaseAuthorized": False,
}
(RUN / "setup-receipt.json").write_text(json.dumps(setup_receipt, indent=2) + "\n")
print("Verified isolated native libraries:", len(setup_receipt["extractedNativeLibraries"]))

## 8. Verify readiness

Readiness compiles a CPU runtime probe, checks pinned assets and queries the CUDA provider. The full graph test in the next cell is still necessary. Both failed and successful receipts are retained. Logs are saved in the evidence folder rather than flooding the notebook.

In [ ]:
BENCHMARK = REPO / "tools/benchmark_deux_accelerator.py"
assert digest(BENCHMARK) == "0c19caa398c6fb1e1be121d359219074e71879979105d4172fbcf360906c9f0a"
base_arguments = [sys.executable, str(BENCHMARK), "--toolchain", str(TOOLCHAIN), "--models", str(MODELS), "--audio", str(AUDIO)]
def run_stage(name, extra):
    output = RUN / name
    assert not output.exists(), "Evidence exists; run setup again for a new run directory."
    with (RUN / (name + ".log")).open("w") as log:
        process = subprocess.Popen(base_arguments + ["--output", str(output)] + extra,
                                   cwd=REPO, env=benchmark_env, stdout=log, stderr=subprocess.STDOUT,
                                   start_new_session=True)
        last_progress = None
        try:
            while process.poll() is None:
                try:
                    process.wait(timeout=20)
                except subprocess.TimeoutExpired:
                    try:
                        partial = json.loads((output / "receipt.json").read_text())
                        progress = (partial.get("status"), len(partial.get("runs", [])))
                    except (FileNotFoundError, json.JSONDecodeError):
                        progress = ("STARTING", 0)
                    if progress != last_progress:
                        print(name, "status:", progress[0], "completed arms/runs:", progress[1], flush=True)
                        last_progress = progress
        except KeyboardInterrupt:
            # Stop the benchmark and its Java children, retaining incomplete evidence.
            try:
                os.killpg(process.pid, signal.SIGTERM)
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
                process.wait()
            except ProcessLookupError:
                process.wait()
            (RUN / (name + "-interrupted.json")).write_text(json.dumps({
                "status": "INTERRUPTED", "completed": False, "target75Proven": False}) + "\n")
            raise
    receipt_path = output / "receipt.json"
    receipt = json.loads(receipt_path.read_text()) if receipt_path.exists() else {"status": "PROCESS_FAILED_WITHOUT_RECEIPT"}
    print(name, "exit:", process.returncode, "status:", receipt["status"])
    if receipt.get("failure"):
        print(receipt["failure"])
    return receipt

readiness = run_stage("readiness", ["--check-readiness"])
assert readiness["status"] == "PREFLIGHT_READY", "Readiness failed. Preserve the evidence; do not substitute CPU execution."

## 9. Run all four diagnostic arms

This runs current CPU/ALL, CPU/BASIC, GPU-package CPU/BASIC and GPU-package CUDA/BASIC, each with an unprofiled and separately profiled passage. CUDA uses `use_tf32=0`. Every graph must execute substantive CUDA arithmetic, while any CPU fallback is reported.

**Expected interpretation:** prior CPU ALL/BASIC controls produced tiny Float32 differences. The existing conservative experiment therefore may finish as `NUMERICAL_EQUIVALENCE_UNPROVEN`. All four diagnostic arms still run before this gate, so CUDA placement and raw numerical differences are useful evidence. Different bits alone are not proof of audible degradation. The gate must not be bypassed to print a speedup.

Repeated, alternating measurements run only if complete output bytes qualify. Even then, the result is a host passage measurement, not a complete song or Android result. Each Java arm has the existing 20-minute timeout; stop and save evidence if the free session cannot complete.

In [ ]:
assert readiness["status"] == "PREFLIGHT_READY"
experiment = run_stage("qualification", ["--warmups", "1", "--repeats", "3"])
summary_keys = ("status", "measured", "outputBytesQualifiedForTiming", "inputsRecheckedAfterQualification",
                "qualityApproved", "target75Proven", "wholeSongSpeedupProven", "androidSpeedupProven", "releaseAuthorized")
print(json.dumps({key: experiment.get(key) for key in summary_keys}, indent=2))
for variant, placement in experiment.get("placement", {}).items():
    print(variant, placement)
for row in experiment.get("comparisons", []):
    if not str(row.get("candidate", "")).endswith("_profiled"):
        print(row["reference"], "→", row["candidate"], "exact:", row["byteIdentical"],
              "max absolute:", row["maximumAbsoluteError"], "RMSE:", row["rootMeanSquaredError"])
if experiment.get("performance"):
    print("Qualified host passage timing only:")
    print(json.dumps(experiment["performance"], indent=2))

## 10. Download the evidence

Run this cell after success, a failed gate or an interrupted experiment. It includes only this run's receipts, logs, source snapshots, traces and public-demo outputs. It excludes model weights, the APK, environment variables, credentials and private files. A partial run is labeled incomplete by its saved receipt; it is never promoted to a result.

Colab provides a download prompt; Kaggle exposes the ZIP in its file browser/output. Download before ending the runtime. To retry, rerun from the first cell: verified downloads can be reused, and a new evidence directory is created.

In [ ]:
archive_path = RUN.with_suffix(".zip")
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1) as archive:
    for path in sorted(RUN.rglob("*")):
        if path.is_file() and not path.is_symlink():
            archive.write(path, arcname=str(path.relative_to(RUN.parent)))
print("Evidence:", archive_path.name, "bytes:", archive_path.stat().st_size)
print("SHA-256:", digest(archive_path))
try:
    from google.colab import files
except ImportError:
    from IPython.display import FileLink, display
    display(FileLink(str(archive_path.relative_to(Path.cwd())) if archive_path.is_relative_to(Path.cwd()) else str(archive_path)))
else:
    files.download(str(archive_path))

## What this experiment can decide

- **CUDA placement succeeds:** the actual free GPU can execute the original graph workload; compare complete outputs and inspect retained CPU fallback.
- **Numerical equivalence is unproven:** inspect the raw differences and run the existing quality process before accepting an optimization. Do not weaken the gate or label its diagnostic times a speedup.
- **Qualified timing succeeds:** retain it as a host passage result. Next measure the complete separation stage and full vocal pipeline, including transfer and startup overhead, against a clean complete-analysis baseline.
- **Runtime or quota blocks execution:** preserve the failure receipt and use another ordinarily available free session or the owned MSI GPU. No paid allocation or reliability guarantee is implied.

**Validation status of this saved template:** notebook format and code compilation are checked locally; public asset recovery and CPU runtime checks are validated separately. A top-to-bottom Colab/Kaggle GPU execution is still required. There are deliberately no fabricated saved GPU outputs.